In [1]:
#!pip install rasterio


In [ ]:
from tqdm import tqdm
import glob
import rasterio
import numpy as np
import json
import os

output_tif = "/data/integracar/amostras_car_mask/"
os.makedirs(output_tif, exist_ok=True)

# Dicionário de cores (exemplo inicial, cores aproximadas)
legend_rgb = {
    (150, 150, 150): 0, #Afloramento Rochoso
    (251, 154, 153): 0, #Área Edificada
    (69, 175, 213): 0, #Brejo
    (150, 109, 207): 3,#Campo Rupestre/Altitude
    (128, 214, 16): 4, #Cultivo Agrícola - Abacaxi
    (247, 223, 8): 4, #Cultivo Agrícola - Banana
    (119, 9, 29): 4, #Cultivo Agrícola - Café
    (209, 163, 117): 4, #Cultivo Agrícola - Cana-De-Açúcar
    (231, 67, 97): 4, #Cultivo Agrícola - Coco-Da-Baía
    (245, 141, 23): 4, #Cultivo Agrícola - Mamão
    (55, 196, 201): 4, #Cultivo Agrícola - Outros Cultivos Permanentes
    (225, 175, 38): 4, #Cultivo Agrícola - Outros Cultivos Temporários
    (81, 77, 77): 5, #Extração Mineração
    (211, 127, 122): 6, #Macega
    (156, 68, 203): 7, #Mangue
    (133, 196, 221): 8, #Massa D'Água
    (13, 103, 19): 9, #Mata Nativa
    (51, 160, 44): 9, #Mata Nativa em Estágio Inicial de Regeneração
    (31, 205, 170): 10, #Outros
    (178, 214, 32): 11, #Pastagem
    (207, 103, 65): 12, #Reflorestamento - Eucalipto
    (243, 184, 129): 12, #Reflorestamento - Pinus
    (151, 132, 233): 12, #Reflorestamento - Seringueira
    (63, 231, 161): 13, #Restinga
    (245, 222, 193): 11 #Solo Exposto
}

# Dicionário reverso para salvar legenda legível
legend_names = {
    0: "Afloramento Rochoso",
    1: "Área Edificada",
    2: "Brejo",
    3: "Campo Rupestre/Altitude",
    4: "Áreas de Cultivo",
    5: "Extração Mineração",
    6: "Macega",
    7: "Mangue",
    8: "Massa D'Água",
    9: "Mata Nativa",
    10: "Outros",
    11: "Solo Exposto",
    12: "Reflorestamento",
    13: "Restinga"
}

# Ler imagem
mask_list = glob.glob('/data/integracar/amostras_car/*.tif')

for mask_path in tqdm(mask_list, desc="Processando imagens"):
    with rasterio.open(mask_path) as src:
        img = src.read()  # (C, H, W)
        profile = src.profile

    # Converter para (H, W, 3)
    img_rgb = np.transpose(img[:3], (1, 2, 0))

    # Criar raster vazio
    class_map = np.zeros((img_rgb.shape[0], img_rgb.shape[1]), dtype=np.uint8)

    # Converter cores → IDs
    for rgb, class_id in legend_rgb.items():
        mask = np.all(img_rgb == rgb, axis=-1)
        class_map[mask] = class_id

    # Atualizar perfil para 1 banda
    profile.update(dtype=rasterio.uint8, count=1)

    base_name = os.path.splitext(os.path.basename(mask_path))[0]
    output_path = os.path.join(output_tif, f'{base_name}.tif')

    # Salvar raster classificado
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(class_map, 1)

    #print("✅ Raster classificado salvo em:", output_path)

Processando imagens: 100%|██████████| 10000/10000 [04:08<00:00, 40.24it/s]


In [ ]:
with rasterio.open(output_path) as src:
    arr = src.read(1)          # [H, W]
    print(arr.min(), arr.max(), np.unique(arr))

terminou


In [3]:
from zipfile import compressor_names

from tqdm import tqdm
import glob
import rasterio
import numpy as np
import json
import os

# from count_classes import legend_rgb

output_tif = "/data/integracar/replicate_article/masks_replicated_full/"

def convert_pixels(convert_type):
    if convert_type == "full_classes":
        legend_rgb = {
            (150, 150, 150): 0, #Afloramento Rochoso
            (251, 154, 153): 1, #Área Edificada
            (69, 175, 213): 2, #Brejo
            (150, 109, 207): 3,#Campo Rupestre/Altitude
            (128, 214, 16): 4, #Cultivo Agrícola - Abacaxi
            (247, 223, 8): 4, #Cultivo Agrícola - Banana
            (119, 9, 29): 4, #Cultivo Agrícola - Café
            (209, 163, 117): 4, #Cultivo Agrícola - Cana-De-Açúcar
            (231, 67, 97): 4, #Cultivo Agrícola - Coco-Da-Baía
            (245, 141, 23): 4, #Cultivo Agrícola - Mamão
            (55, 196, 201): 4, #Cultivo Agrícola - Outros Cultivos Permanentes
            (225, 175, 38): 4, #Cultivo Agrícola - Outros Cultivos Temporários
            (81, 77, 77): 5, #Extração Mineração
            (211, 127, 122): 2, #Macega
            (156, 68, 203): 2, #Mangue
            (133, 196, 221): 6, #Massa D'Água
            (13, 103, 19): 7, #Mata Nativa
            (51, 160, 44): 7, #Mata Nativa em Estágio Inicial de Regeneração
            (31, 205, 170): 8, #Outros
            (178, 214, 32): 9, #Pastagem
            (207, 103, 65): 10, #Reflorestamento - Eucalipto
            (243, 184, 129): 10, #Reflorestamento - Pinus
            (151, 132, 233): 10, #Reflorestamento - Seringueira
            (63, 231, 161): 2, #Restinga
            (245, 222, 193): 9 #Solo Exposto
        }

        legend_names = {
            0: "Afloramento Rochoso",
            1: "Área Edificada",
            2: "Áreas de Vegetação",
            3: "Campo Rupestre/Altitude",
            4: "Áreas de Cultivo",
            5: "Extração Mineração",
            6: "Massa D'Água",
            7: "Mata Nativa",
            8: "Outros",
            9: "Solo Exposto",
            10: "Reflorestamento"
        }

        return legend_names, legend_rgb

    elif convert_type=="binary_forest":
        legend_rgb = {
            (150, 150, 150): 0, #Afloramento Rochoso
            (251, 154, 153): 0, #Área Edificada
            (69, 175, 213): 0, #Brejo
            (150, 109, 207): 0,#Campo Rupestre/Altitude
            (128, 214, 16): 1, #Cultivo Agrícola - Abacaxi
            (247, 223, 8): 1, #Cultivo Agrícola - Banana
            (119, 9, 29): 1, #Cultivo Agrícola - Café
            (209, 163, 117): 1, #Cultivo Agrícola - Cana-De-Açúcar
            (231, 67, 97): 1, #Cultivo Agrícola - Coco-Da-Baía
            (245, 141, 23): 1, #Cultivo Agrícola - Mamão
            (55, 196, 201): 1, #Cultivo Agrícola - Outros Cultivos Permanentes
            (225, 175, 38): 1, #Cultivo Agrícola - Outros Cultivos Temporários
            (81, 77, 77): 0, #Extração Mineração
            (211, 127, 122): 1, #Macega
            (156, 68, 203): 1, #Mangue
            (133, 196, 221): 0, #Massa D'Água
            (13, 103, 19): 1, #Mata Nativa
            (51, 160, 44): 1, #Mata Nativa em Estágio Inicial de Regeneração
            (31, 205, 170): 0, #Outros
            (178, 214, 32): 1, #Pastagem
            (207, 103, 65): 1, #Reflorestamento - Eucalipto
            (243, 184, 129): 1, #Reflorestamento - Pinus
            (151, 132, 233): 1, #Reflorestamento - Seringueira
            (63, 231, 161): 1, #Restinga
            (245, 222, 193): 0 #Solo Exposto
        }

        legend_names = {
            0: "sem_vegetacao",
            1: "com_vegetacao",
            2: "nulos"
        }

        return legend_names, legend_rgb

    else:
        raise TypeError("Tipo de conversão não definido!!!")


mask_list = glob.glob('/data/integracar/replicate_article/mask_images/*.tif')

legend_names, legend_rgb = convert_pixels(convert_type="full_classes")

for mask_path in tqdm(mask_list, desc="Processando imagens"):
    with rasterio.open(mask_path) as src:
        img = src.read()  # (C, H, W)
        profile = src.profile

    # Converter para (H, W, 3)
    img_rgb = np.transpose(img[:3], (1, 2, 0))

    # Criar raster vazio
    # class_map = np.zeros((img_rgb.shape[0], img_rgb.shape[1]), dtype=np.uint8)
    class_map = np.full((img_rgb.shape[0], img_rgb.shape[1]), 11, dtype=np.uint8) # para binário usar 2, para full usar 11

    # Converter cores → IDs
    for rgb, class_id in legend_rgb.items():
        mask = np.all(img_rgb == rgb, axis=-1)
        class_map[mask] = class_id

    # Atualizar perfil para 1 banda
    profile.update(dtype=rasterio.uint8, count=1)

    base_name = os.path.splitext(os.path.basename(mask_path))[0]
    output_path = os.path.join(output_tif, f'{base_name}.tif')

    # Salvar raster classificado
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(class_map, 1)

    #print("✅ Raster classificado salvo em:", output_path)

Processando imagens: 100%|██████████| 500/500 [00:43<00:00, 11.61it/s]
